In [0]:
df_txn = spark.table("workspace.gold_sales_bronze.raw_transactions")

In [0]:
from pyspark.sql.types import StructType, StringType, DoubleType, BooleanType, TimestampType

audit_schema = StructType() \
    .add("transaction_id", StringType()) \
    .add("assay_certificate", StringType()) \
    .add("assay_lab", StringType()) \
    .add("purity_verified", StringType()) \
    .add("weight_oz", DoubleType()) \
    .add("vault_location", StringType()) \
    .add("inspection_timestamp", TimestampType()) \
    .add("inspector_id", StringType()) \
    .add("insurance_policy", StringType()) \
    .add("flags", StructType() \
        .add("suspicious", BooleanType()) \
        .add("manual_review", BooleanType())
    )

In [0]:
from pyspark.sql.functions import col, from_json

df_txn = spark.table("workspace.gold_sales_bronze.raw_transactions")

df_parsed = df_txn \
    .withColumn("audit_parsed", from_json(col("audit_metadata_json"), audit_schema)) \
    .select(
        "transaction_id",
        col("audit_parsed.assay_certificate"),
        col("audit_parsed.assay_lab"),
        col("audit_parsed.purity_verified"),
        col("audit_parsed.weight_oz"),
        col("audit_parsed.vault_location"),
        col("audit_parsed.inspection_timestamp"),
        col("audit_parsed.inspector_id"),
        col("audit_parsed.insurance_policy"),
        col("audit_parsed.flags.suspicious").alias("flag_suspicious"),
        col("audit_parsed.flags.manual_review").alias("flag_manual_review"),
        col("audit_parsed.transaction_id").alias("assay_txn_id")
    )

In [0]:
from pyspark.sql.functions import when

df_final = df_parsed \
    .withColumn(
        "is_valid_txn",
        col("assay_txn_id") == col("transaction_id")
    ) \
    .withColumn(
        "is_suspicious",
        when(col("flag_suspicious") == True, True).otherwise(False)
    )

In [0]:
target_table = "workspace.gold_sales_silver.flattened_audit"

if spark.catalog.tableExists(target_table):
    
    existing = spark.table(target_table).select("transaction_id").distinct()
    
    df_final = df_final.join(existing, on="transaction_id", how="left_anti")

    df_final.write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

else:
    df_final.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)